In [0]:
import pandas as pd
import json
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DateType, ArrayType
import re
import os

In [0]:
# initial and last date
initial_date = datetime.strptime(dbutils.widgets.get("start_date"), '%Y%m%d')
last_date = datetime.strptime(datetime.now().strftime("%Y%m%d"), '%Y%m%d')

# generate a set of all dates between initial_date and last_date
all_dates = set(initial_date + timedelta(days=x) for x in range((last_date - initial_date).days + 1))

# select only the start of each quarter
quarter_start_months = [1, 4, 7, 10]
quarter_start_dates = [
    date for date in sorted(all_dates)
    if date.month in quarter_start_months and date.day == 1 and date >= initial_date and date <= last_date
]

# convert to string format
historical_dates = [date.strftime('%Y%m%d') for date in quarter_start_dates]

# replace 20240201 with 20240202
historical_dates = ['20240202' if date == '20240201' else date for date in historical_dates]
historical_dates = ['20240702' if date == '20240701' else date for date in historical_dates]

# convert the list to a jason array
date_json_array = json.dumps(historical_dates, indent=4)

# making it available for using it in workflow with key 'dates_list'
dbutils.jobs.taskValues.set(key = 'dates_list', value = date_json_array)

In [0]:
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
ServiceKey='app-reg-databricks-wr-radar-preprd'
service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")

spark.conf.set("fs.azure.account.auth.type."+ReadStorage+".dfs.core.windows.net", "OAuth") 
spark.conf.set("fs.azure.account.oauth.provider.type."+ReadStorage+".dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider") 
spark.conf.set("fs.azure.account.oauth2.client.id."+ReadStorage+".dfs.core.windows.net", ""+app_reg_app_id+"") 
spark.conf.set("fs.azure.account.oauth2.client.secret."+ReadStorage+".dfs.core.windows.net", service_credential) 
spark.conf.set("fs.azure.account.oauth2.client.endpoint."+ReadStorage+".dfs.core.windows.net", "https://login.microsoftonline.com/"+TenantId+"/oauth2/token") 

In [0]:
# define the date format
date_format = "%Y%m%d"

# define gdp version
version = 100 #1 EBX

def process_gdp_object(gdp_object):

    path = dbutils.fs.ls(f'abfss://gcob@{ReadStorage}.dfs.core.windows.net/{gdp_object}/{version}/data/')

    # path = dbutils.fs.ls(f'abfss://ebx@{ReadStorage}.dfs.core.windows.net/{gdp_object}/{version}/data/') # EBX

    # extract all dates from path
    dates_in_path = [datetime.strptime(re.split(r"[=,/]", f.name)[1][:8], date_format) for f in path]
    
    # sort dates_in_path
    dates_in_path.sort()
    
    # convert dates_in_path to str format to be compares with historical_dates
    dates_in_path_str = {date.strftime(date_format) for date in dates_in_path}

    missing_historical_dates = sorted(set(historical_dates) - set(dates_in_path_str))
    
    # name of gdp_object, and missing dates wrt to historical_dates
    result = (gdp_object, missing_historical_dates)
    
    return result


gcob_gdp_objects = [
    'CaseService_case_Case'
    , 'RiskModel_dbo_InstanceAnswer'
    , 'RiskModel_dbo_PossibleAnswer'
    , 'CaseService_dbo_CddType'
    , 'CaseService_case_DynamicRiskModelInstanceReference'
    , 'CaseService_NaturalPerson_DynamicRiskModelInstanceReference'
    , 'CaseService_dbo_Country'
    , 'CaseService_case_HighRiskProductReference'
    , 'CaseService_NaturalPerson_HighRiskProductReference'
    , 'CaseService_case_LegalEntityClientStructureSnapshot'
    , 'CaseService_snapshot_ClientStructureSnapshotRelationshipDetail'
    , 'CaseService_snapshot_ClientStructureSnapshotCalculatedUboShareholdingDetail'
    , 'CaseService_case_RelatedNaturalPersonParty'
    , 'CaseService_case_PoliticallyExposedPersonStatusReference'
    , 'CaseService_case_RelatedNaturalPersonPartyNationality'
    , 'CaseService_case_Address'
    , 'CaseService_case_ScreeningResults'
    , 'CaseService_case_ProductProvidedToLegalEntity'
    , 'CaseService_dbo_RabobankEntity'
    , 'CaseService_dbo_Product'
    , 'CaseService_NaturalPerson_ProductProvidedToNaturalPerson'
    , 'CaseService_case_LegalEntityBusinessActivity'
    , 'CaseService_dbo_Naics'
    , 'CaseService_NaturalPerson_NaturalPersonBusinessActivity'
    , 'CaseService_case_LegalEntityClient'
    
    # 'EBX_FECCountryRiskList'
]

# process each gdp_object and collect results
results = [process_gdp_object(g) for g in gcob_gdp_objects]

# define schema for df
schema = StructType([
    StructField('gdp_object', StringType(), True),
    StructField('missing_dates', ArrayType(StringType()), True)
])

# create df from results
df = spark.createDataFrame(results, schema=schema)

df.display()

In [0]:
from pyspark.sql.functions import explode

distinct_dates = df.select(explode("missing_dates").alias("date")) \
                   .distinct() \
                   .rdd.flatMap(lambda x: x) \
                   .collect()
distinct_dates = sorted(distinct_dates)
distinct_dates